# 🎮 Esports in Italia: Dal Covid ad Oggi
**Data Jam – 27 maggio 2026**

Domanda principale:
> Gli esports sono cresciuti in Italia dal Covid ad oggi? Come si confronta con il resto del mondo?

Dataset usati:
- `GeneralEsportData.csv` – dati totali per gioco
- `HistoricalEsportData.csv` – dati mese per mese dal 1998 al 2024
- Fonte dati italiani: IIDEA – Italian Interactive Digital Entertainment Association


## 📦 1. Importa le Librerie

In [ ]:
# Librerie per analisi dati
import pandas as pd
import numpy as np

# Librerie per grafici
import matplotlib.pyplot as plt
import seaborn as sns

# Librerie per ML
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Impostazioni grafici
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("✅ Tutte le librerie caricate!")


## 📂 2. Carica i Dati

In [ ]:
# Carica i due CSV
gen = pd.read_csv('GeneralEsportData.csv')
hist = pd.read_csv('HistoricalEsportData.csv')

# Converte la colonna Date in formato data
hist['Date'] = pd.to_datetime(hist['Date'])

print("✅ GeneralEsportData:", gen.shape[0], "giochi,", gen.shape[1], "colonne")
print("✅ HistoricalEsportData:", hist.shape[0], "righe,", hist.shape[1], "colonne")
print()
print("📅 Periodo disponibile:", hist['Date'].min().strftime('%Y-%m'), "→", hist['Date'].max().strftime('%Y-%m'))


### Prima occhiata ai dati

In [ ]:
print("=== GeneralEsportData (prime 3 righe) ===")
display(gen.head(3))

print("\n=== HistoricalEsportData (prime 3 righe) ===")
display(hist.head(3))


## 🧹 3. Prepara i Dati (Dal Covid in Poi)

Filtriamo solo i dati dal **2019 in poi** — così includiamo il periodo pre-Covid (2019) e tutto il periodo Covid e post-Covid.

In [ ]:
# Filtra solo dal 2019 in poi
hist_covid = hist[hist['Date'] >= '2019-01-01'].copy()

# Aggiunge colonna Anno
hist_covid['Year'] = hist_covid['Date'].dt.year

print("✅ Righe dopo il 2019:", len(hist_covid))
print("✅ Anni disponibili:", sorted(hist_covid['Year'].unique()))


## 📈 4. Crescita degli Esports nel Tempo

### Domanda: I premi dei tornei sono cresciuti dal Covid ad oggi?

Raggruppiamo i dati per **anno** e sommiamo tutti i guadagni.


In [ ]:
# Somma guadagni per anno
earnings_per_year = hist_covid.groupby('Year')['Earnings'].sum().reset_index()
earnings_per_year.columns = ['Anno', 'Guadagni_Totali']

# Converte in milioni per leggibilità
earnings_per_year['Guadagni_Milioni'] = earnings_per_year['Guadagni_Totali'] / 1_000_000

print(earnings_per_year[['Anno', 'Guadagni_Milioni']].to_string(index=False))


In [ ]:
# GRAFICO 1 – Guadagni totali per anno
fig, ax = plt.subplots()

colors = ['#e74c3c' if y == 2020 else '#3498db' for y in earnings_per_year['Anno']]
bars = ax.bar(earnings_per_year['Anno'].astype(str), earnings_per_year['Guadagni_Milioni'], color=colors, edgecolor='white')

# Etichette sopra le barre
for bar, val in zip(bars, earnings_per_year['Guadagni_Milioni']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'${val:.0f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('💰 Guadagni Totali Esports per Anno (2019–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Anno')
ax.set_ylabel('Guadagni (Milioni $)')
ax.annotate('📌 Covid-19', xy=('2020', earnings_per_year[earnings_per_year['Anno']==2020]['Guadagni_Milioni'].values[0]),
            xytext=(1.5, earnings_per_year['Guadagni_Milioni'].max() * 0.85),
            fontsize=10, color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))
plt.tight_layout()
plt.savefig('grafico1_guadagni_per_anno.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


### Domanda: Il numero di tornei e giocatori è aumentato?

In [ ]:
# Raggruppa tornei e giocatori per anno
activity = hist_covid.groupby('Year').agg(
    Tornei=('Tournaments', 'sum'),
    Giocatori=('Players', 'sum')
).reset_index()

display(activity)


In [ ]:
# GRAFICO 2 – Tornei e Giocatori per anno (doppio grafico)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico tornei
ax1.plot(activity['Year'], activity['Tornei'], marker='o', color='#9b59b6', linewidth=2.5, markersize=8)
ax1.fill_between(activity['Year'], activity['Tornei'], alpha=0.2, color='#9b59b6')
ax1.set_title('🏆 Tornei per Anno', fontweight='bold')
ax1.set_xlabel('Anno')
ax1.set_ylabel('Numero di Tornei')
ax1.axvline(x=2020, color='red', linestyle='--', alpha=0.5, label='Covid')
ax1.legend()

# Grafico giocatori
ax2.plot(activity['Year'], activity['Giocatori'], marker='s', color='#2ecc71', linewidth=2.5, markersize=8)
ax2.fill_between(activity['Year'], activity['Giocatori'], alpha=0.2, color='#2ecc71')
ax2.set_title('👾 Giocatori per Anno', fontweight='bold')
ax2.set_xlabel('Anno')
ax2.set_ylabel('Numero di Giocatori')
ax2.axvline(x=2020, color='red', linestyle='--', alpha=0.5, label='Covid')
ax2.legend()

plt.suptitle('Crescita degli Esports nel Mondo (2019–2024)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico2_tornei_giocatori.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## 🎮 5. I Giochi Più Popolari

### Domanda: Quali giochi guadagnano di più in totale?

Usiamo `GeneralEsportData.csv` per vedere la classifica storica.


In [ ]:
# Top 10 giochi per guadagni totali
top10_earnings = gen.nlargest(10, 'TotalEarnings')[['Game', 'TotalEarnings', 'Genre']].copy()
top10_earnings['TotalEarnings_M'] = top10_earnings['TotalEarnings'] / 1_000_000
top10_earnings = top10_earnings.sort_values('TotalEarnings_M')

# GRAFICO 3 – Top 10 giochi
fig, ax = plt.subplots(figsize=(12, 6))

palette = sns.color_palette("viridis", 10)
bars = ax.barh(top10_earnings['Game'], top10_earnings['TotalEarnings_M'], color=palette)

for bar, val in zip(bars, top10_earnings['TotalEarnings_M']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'${val:.0f}M', va='center', fontsize=9, fontweight='bold')

ax.set_title('🎮 Top 10 Giochi per Guadagni Totali (storia completa)', fontsize=14, fontweight='bold')
ax.set_xlabel('Guadagni Totali (Milioni $)')
plt.tight_layout()
plt.savefig('grafico3_top10_giochi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


### Domanda: Quale genere di gioco domina gli esports?

In [ ]:
# Guadagni per genere
genre_earnings = gen.groupby('Genre')['TotalEarnings'].sum().sort_values(ascending=False) / 1_000_000

# GRAFICO 4 – Guadagni per genere
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Set2", len(genre_earnings))
bars = ax.bar(genre_earnings.index, genre_earnings.values, color=colors, edgecolor='white')

ax.set_title('🕹️ Guadagni per Genere di Gioco', fontsize=14, fontweight='bold')
ax.set_xlabel('Genere')
ax.set_ylabel('Guadagni Totali (Milioni $)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('grafico4_generi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## 🌍 6. Quali Giochi Hanno Dominato Durante il Covid?

### Domanda: Durante il Covid (2020–2021), quali giochi erano più popolari?


In [ ]:
# Giochi più guadagnati nel periodo Covid (2020-2021)
covid_period = hist_covid[hist_covid['Year'].isin([2020, 2021])]
top_covid_games = covid_period.groupby('Game')['Earnings'].sum().nlargest(10).sort_values() / 1_000_000

# GRAFICO 5 – Top giochi durante Covid
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette("rocket", 10)
bars = ax.barh(top_covid_games.index, top_covid_games.values, color=colors)

for bar, val in zip(bars, top_covid_games.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'${val:.1f}M', va='center', fontsize=9, fontweight='bold')

ax.set_title('🏠 Top 10 Giochi durante il Covid (2020–2021)', fontsize=14, fontweight='bold')
ax.set_xlabel('Guadagni (Milioni $)')
plt.tight_layout()
plt.savefig('grafico5_giochi_covid.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## 🇮🇹 7. Italia vs Mondo

I CSV di Kaggle non hanno una colonna "paese", quindi per i **dati italiani** usiamo i numeri ufficiali dal report IIDEA 2024.

> **Fonte:** IIDEA – 2024 Italian Esports Report (Market & Streaming Trends)


In [ ]:
# Dati IIDEA italiani (manuali dal PDF ufficiale)
italia_data = {
    'Anno': [2020, 2021, 2022, 2023, 2024],
    'Fan_Milioni': [5.8, 6.5, 5.9, 6.8, 7.3],  # milioni di fan esports in Italia
    'Nota': ['Boom Covid', 'Crescita', 'Calo post-Covid (-10%)', 'Ripresa', 'Record']
}
df_italia = pd.DataFrame(italia_data)
display(df_italia)


In [ ]:
# Dati mondiali: guadagni totali per anno (dai nostri CSV)
world_data = hist_covid.groupby('Year')['Earnings'].sum().reset_index()
world_data.columns = ['Anno', 'Guadagni_Mondo_M']
world_data['Guadagni_Mondo_M'] = world_data['Guadagni_Mondo_M'] / 1_000_000

# GRAFICO 6 – Italia Fan vs Mondo Guadagni (doppio asse)
fig, ax1 = plt.subplots(figsize=(12, 6))

color_ita = '#009246'  # verde Italia
color_world = '#3498db'

# Asse sinistra: Fan Italia
ax1.bar(df_italia['Anno'].astype(str), df_italia['Fan_Milioni'],
        color=color_ita, alpha=0.7, label='🇮🇹 Fan Esports Italia (milioni)', width=0.4)
ax1.set_xlabel('Anno')
ax1.set_ylabel('Fan Esports in Italia (Milioni)', color=color_ita)
ax1.tick_params(axis='y', labelcolor=color_ita)

# Asse destra: Guadagni mondo
ax2 = ax1.twinx()
ax2.plot(world_data['Anno'].astype(str), world_data['Guadagni_Mondo_M'],
         marker='o', color=color_world, linewidth=2.5, markersize=8, label='🌍 Guadagni Mondo ($M)')
ax2.set_ylabel('Guadagni Mondiali (Milioni $)', color=color_world)
ax2.tick_params(axis='y', labelcolor=color_world)

# Legenda combinata
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('🇮🇹 Italia vs 🌍 Mondo — Crescita Esports (2019–2024)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico6_italia_vs_mondo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## 🤖 8. Modello ML – Clustering dei Giochi

### Cosa facciamo?
Raggruppiamo i giochi in **cluster** (gruppi) in base a:
- Guadagni totali
- Numero di giocatori
- Numero di tornei

**K-Means Clustering** = un algoritmo che trova gruppi simili automaticamente.

📦 Usa `GeneralEsportData.csv`


In [ ]:
# Prepara i dati per il clustering
features = gen[['TotalEarnings', 'TotalPlayers', 'TotalTournaments']].copy()

# Normalizza i dati (importante per K-Means!)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Applica K-Means con 3 cluster
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
gen['Cluster'] = kmeans.fit_predict(features_scaled)

# Risultati
cluster_summary = gen.groupby('Cluster').agg(
    Num_Giochi=('Game', 'count'),
    Guadagni_Medi_M=('TotalEarnings', lambda x: round(x.mean()/1_000_000, 1)),
    Giocatori_Medi=('TotalPlayers', lambda x: round(x.mean(), 0)),
    Tornei_Medi=('TotalTournaments', lambda x: round(x.mean(), 0))
).reset_index()

cluster_summary['Cluster'] = cluster_summary['Cluster'].map({
    0: '🟡 Cluster 0 – Piccoli',
    1: '🟢 Cluster 1 – Medi',
    2: '🔴 Cluster 2 – Grandi'
})

display(cluster_summary)


In [ ]:
# GRAFICO 7 – Clustering scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

colors_cluster = {0: '#f39c12', 1: '#2ecc71', 2: '#e74c3c'}
labels_cluster = {0: '🟡 Piccoli', 1: '🟢 Medi', 2: '🔴 Grandi (Élite)'}

for c in [0, 1, 2]:
    mask = gen['Cluster'] == c
    ax.scatter(
        gen[mask]['TotalPlayers'],
        gen[mask]['TotalEarnings'] / 1_000_000,
        label=labels_cluster[c],
        color=colors_cluster[c],
        alpha=0.7, s=60
    )

# Etichetta i top giochi
top_labels = gen.nlargest(5, 'TotalEarnings')
for _, row in top_labels.iterrows():
    ax.annotate(row['Game'], (row['TotalPlayers'], row['TotalEarnings']/1_000_000),
                fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Numero di Giocatori')
ax.set_ylabel('Guadagni Totali (Milioni $)')
ax.set_title('🤖 Clustering dei Giochi Esports (K-Means)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('grafico7_clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## 📉 9. Modello ML – Regressione Lineare

### Cosa facciamo?
Usiamo una **regressione lineare** per prevedere i guadagni futuri degli esports nel mondo.

**Regressione lineare** = trova una linea che segue i dati e la estende nel futuro.


In [ ]:
# Dati annuali dal 2019
yearly = hist_covid.groupby('Year')['Earnings'].sum().reset_index()
yearly.columns = ['Year', 'Earnings']
yearly['Earnings_M'] = yearly['Earnings'] / 1_000_000

# Addestramento modello
X = yearly[['Year']]
y = yearly['Earnings_M']

model = LinearRegression()
model.fit(X, y)

# Predizione 2025 e 2026
future_years = pd.DataFrame({'Year': [2025, 2026]})
predictions = model.predict(future_years)

r2 = r2_score(y, model.predict(X))
print(f"R² score: {r2:.3f} (più vicino a 1.0 = migliore)")
print(f"Previsione 2025: ${predictions[0]:.0f}M")
print(f"Previsione 2026: ${predictions[1]:.0f}M")


In [ ]:
# GRAFICO 8 – Regressione
fig, ax = plt.subplots(figsize=(10, 5))

# Dati reali
ax.scatter(yearly['Year'], yearly['Earnings_M'], color='#3498db', s=80, zorder=5, label='Dati reali')
ax.plot(yearly['Year'], yearly['Earnings_M'], color='#3498db', linestyle='--', alpha=0.5)

# Linea di regressione estesa
all_years = list(range(2019, 2027))
all_pred = model.predict(pd.DataFrame({'Year': all_years}))
ax.plot(all_years, all_pred, color='#e74c3c', linewidth=2, label='Trend (regressione)')

# Punti futuri
ax.scatter([2025, 2026], predictions, color='#e74c3c', s=100, zorder=5,
           marker='*', label='Previsione 2025-2026')

for yr, pred in zip([2025, 2026], predictions):
    ax.annotate(f'${pred:.0f}M', (yr, pred), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=10, color='#e74c3c')

ax.set_xlabel('Anno')
ax.set_ylabel('Guadagni Mondiali (Milioni $)')
ax.set_title('📉 Previsione Guadagni Esports 2025–2026 (Regressione Lineare)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('grafico8_regressione.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafico salvato!")


## ✅ 10. Conclusioni

### Cosa abbiamo scoperto?

**📈 Crescita mondiale:**
- I guadagni degli esports sono cresciuti in modo costante dal 2019 al 2024
- Il Covid (2020) ha portato un forte aumento — tornei online hanno sostituito quelli fisici
- Nel 2022 c'è stato un piccolo calo post-pandemia, poi ripresa nel 2023-2024

**🎮 Giochi più popolari:**
- **Dota 2**, **Fortnite** e **CS:GO** dominano per guadagni totali
- Il genere **MOBA** (Multiplayer Online Battle Arena) è il più redditizio
- Durante il Covid, i giochi online come Dota 2 e League of Legends erano in cima

**🇮🇹 Italia:**
- Nel 2024 ci sono **7,3 milioni** di fan degli esports in Italia
- L'Italia è ancora piccola rispetto a USA e Corea del Sud, ma **la crescita è alta**
- Covid 2020: boom di interesse → 2022: calo → 2024: nuovo record

**🤖 Machine Learning:**
- Il clustering ha diviso i giochi in 3 gruppi: piccoli, medi, e élite (Dota 2, Fortnite, CS:GO)
- La regressione prevede una crescita continua dei guadagni nel 2025-2026

---
*Fonti: Kaggle Esports Earnings 1998–2023 | IIDEA 2024 Italian Esports Report*
